# 01 — Exploratory data analysis

Quick EDA on `data/raw/data_sensors.csv`:
1. Shape, column types, missingness
2. Label distribution
3. Per-sensor distributions
4. Inter-sensor correlation matrix
5. PCA scree plot
6. UMAP first look (overlay labeled points)

These plots feed the slide deck and the report.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

ROOT = Path("..").resolve()
df = pd.read_csv(ROOT / "data/raw/data_sensors.csv")
print(df.shape)
df.head()

In [ ]:
# 1. Missingness and label distribution
print("NaN counts per column:")
print(df.isna().sum())
print()
print("Label distribution:")
print(df["Label"].value_counts(dropna=False))

In [ ]:
# 2. Per-sensor distributions
fig, axes = plt.subplots(4, 5, figsize=(15, 9))
for i, ax in enumerate(axes.ravel()):
    sns.histplot(df[f"Sensor {i}"], bins=40, ax=ax)
    ax.set_title(f"Sensor {i}")
    ax.set_xlabel("")
fig.suptitle("Per-sensor distributions")
fig.tight_layout()

In [ ]:
# 3. Inter-sensor correlation matrix
X = df.drop(columns=["Label"])
corr = X.corr()
fig, ax = plt.subplots(figsize=(8, 7))
sns.heatmap(corr, cmap="coolwarm", center=0, vmin=-1, vmax=1, ax=ax)
ax.set_title("Sensor correlation matrix")
fig.tight_layout()
print("mean |off-diag corr|:", np.abs(corr.to_numpy()[~np.eye(len(corr), dtype=bool)]).mean())

In [ ]:
# 4. PCA scree plot
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

Xs = StandardScaler().fit_transform(X)
pca = PCA(svd_solver="full").fit(Xs)
evr = pca.explained_variance_ratio_
cum = np.cumsum(evr)

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(range(1, len(evr) + 1), evr, alpha=0.6, label="per-component")
ax.plot(range(1, len(cum) + 1), cum, "o-", color="red", label="cumulative")
ax.axhline(0.95, color="grey", linestyle="--", label="95% variance")
ax.set_xlabel("PCA component")
ax.set_ylabel("explained variance ratio")
ax.set_title("PCA scree")
ax.legend()
fig.tight_layout()
print("Components needed for 95% var:", np.searchsorted(cum, 0.95) + 1)

In [ ]:
# 5. First-look UMAP
import umap

coords = umap.UMAP(n_components=2, random_state=42).fit_transform(Xs)
y = df["Label"].to_numpy()
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(coords[:, 0], coords[:, 1], s=8, c="lightgrey", label="unlabeled")
for cls, marker in zip([1.0, 2.0, 3.0], ["o", "s", "^"], strict=True):
    mask = y == cls
    ax.scatter(
        coords[mask, 0],
        coords[mask, 1],
        s=70,
        marker=marker,
        label=f"CLASS_{int(cls)}",
        edgecolor="black",
        linewidth=1.2,
    )
ax.set_title("UMAP first look — labeled points overlaid")
ax.legend()
fig.tight_layout()